In [1]:
# RAG Pipeline for a Git Repository
#
# This script demonstrates how to build a complete Retrieval-Augmented Generation (RAG)
# pipeline for a software project hosted in a Git repository. It will:
#
# 1.  Clone a Git Repository: Fetch the source code and documentation.
# 2.  Load & Process Data: Load .py and .md files and split them into manageable chunks.
# 3.  Embed & Store: Convert the chunks into vector embeddings and store them in a FAISS vector database.
# 4.  Build RAG Chain: Create a pipeline that takes a question, retrieves relevant documents,
#     and uses an LLM to generate an answer.
#
# To run this script, you first need to install the dependencies:
# pip install langchain langchain-community langchain-huggingface faiss-cpu sentence-transformers GitPython beautifulsoup4

import git
import os
from langchain_community.document_loaders import TextLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFaceEndpoint
from langchain.prompts import PromptTemplate
from langchain.schema.runnable import RunnablePassthrough
from langchain.schema.output_parser import StrOutputParser

def clone_repository(repo_url, repo_path):
    """Clones a Git repository if it doesn't already exist."""
    if not os.path.exists(repo_path):
        print(f"Cloning repository from {repo_url} to {repo_path}...")
        git.Repo.clone_from(repo_url, repo_path)
        print("Cloning complete.")
    else:
        print(f"Repository already exists at {repo_path}.")

def load_documents_from_repository(repo_path):
    """Loads all .py and .md files from the repository."""
    print("Loading documents from the repository...")
    # Create a loader for python files
    py_loader = DirectoryLoader(repo_path, glob="**/*.py", loader_cls=TextLoader, recursive=True, show_progress=True, use_multithreading=True)
    # Create a loader for markdown files
    md_loader = DirectoryLoader(repo_path, glob="**/*.md", loader_cls=TextLoader, recursive=True, show_progress=True, use_multithreading=True)

    py_docs = py_loader.load()
    md_docs = md_loader.load()

    print(f"Loaded {len(py_docs)} Python files and {len(md_docs)} Markdown files.")
    return py_docs + md_docs

def chunk_documents(documents):
    """Splits documents into smaller chunks."""
    print("Chunking documents...")
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
    chunked_documents = text_splitter.split_documents(documents)
    print(f"Created {len(chunked_documents)} chunks.")
    return chunked_documents

def create_and_save_vector_store(chunked_documents, embeddings_model_name, index_path):
    """Creates and saves a FAISS vector store from document chunks."""
    # Use a popular open-source embedding model
    model_kwargs = {'device': 'cpu'} # Use CPU for embedding
    embeddings = HuggingFaceEmbeddings(model_name=embeddings_model_name, model_kwargs=model_kwargs)

    print("Creating vector store... This may take a while.")
    # Create the FAISS vector store from our chunks
    vector_store = FAISS.from_documents(chunked_documents, embeddings)
    print("Vector store created.")

    # Save the vector store locally
    vector_store.save_local(index_path)
    print(f"Vector store saved to {index_path}.")

def create_rag_chain(llm_repo_id, index_path, embeddings_model_name):
    """Creates the RAG chain for question answering."""
    # Load the local vector store
    embeddings = HuggingFaceEmbeddings(model_name=embeddings_model_name)
    vector_store = FAISS.load_local(index_path, embeddings, allow_dangerous_deserialization=True)

    # Set up the retriever
    retriever = vector_store.as_retriever(search_kwargs={"k": 5}) # Retrieve top 5 most relevant chunks

    # Define the LLM
    # You need to have a Hugging Face Hub API token in your environment.
    # You can get one from https://huggingface.co/settings/tokens
    # In your terminal, run: export HF_TOKEN='your_token_here'
    llm = HuggingFaceEndpoint(
        repo_id=llm_repo_id,
        temperature=0.1,
        max_new_tokens=512
    )

    # Define the prompt template
    template = """
You are an expert assistant for the scikit-learn library.
Use the following pieces of context from the codebase to answer the question at the end.
If you don't know the answer, just say that you don't know, don't try to make up an answer.

Context: {context}

Question: {question}

Helpful Answer:
"""
    prompt = PromptTemplate(template=template, input_variables=["context", "question"])

    # Create the RAG chain
    rag_chain = (
        {"context": retriever, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )

    print("RAG chain created successfully.")
    return rag_chain

def ask_question(rag_chain, query):
    """Asks a question to the RAG chain and prints the answer."""
    print(f"\n--- Query: {query} ---")
    answer = rag_chain.invoke(query)
    print("Answer:", answer)
    return answer

def main():
    """Main function to run the RAG pipeline."""
    # Configuration
    repo_url = "https://github.com/scikit-learn/scikit-learn.git"
    repo_path = "./repo/scikit-learn"
    index_path = "faiss_index_sklearn"
    embeddings_model_name = "sentence-transformers/all-MiniLM-L6-v2"
    llm_repo_id = "google/flan-t5-large"

    # --- Step 1: Clone Repository ---
    clone_repository(repo_url, repo_path)

    # --- Step 2 & 3: Load, Chunk, and Create Vector Store ---
    # This part is computationally expensive. We check if the index already exists.
    if not os.path.exists(index_path):
        documents = load_documents_from_repository(repo_path)
        chunks = chunk_documents(documents)
        create_and_save_vector_store(chunks, embeddings_model_name, index_path)
    else:
        print(f"Vector store index found at {index_path}. Skipping creation.")

    # --- Step 4: Create RAG Chain ---
    rag_chain = create_rag_chain(llm_repo_id, index_path, embeddings_model_name)

    # --- Step 5: Ask Questions ---
    ask_question(rag_chain, "What is the main purpose of the TfidfVectorizer?")
    ask_question(rag_chain, "How is the 'gini' impurity calculated in a DecisionTreeClassifier?")
    ask_question(rag_chain, "Can you show me an example of how to use the KMeans clustering algorithm?")

if __name__ == "__main__":
    main()

ModuleNotFoundError: No module named 'git'